# Stage 6 — Evidence Scoring

Scores each candidate diagnosis from **Ontology Routing** (Stage 5) against the documented evidence in the
**symptom tree** and **information extraction** (Stage 3): a 0-100 score, supporting/contradicting/missing
evidence, and a rationale per candidate.

**Input:** `symptom_tree_results.json` (Stage 3) + ontology routing per-patient files (Stage 5)
**Output:** one JSON file per patient in `data/stage_06_evidence_scoring/` + an index


## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

import pandas as pd
from pipeline import (
    LLMNotAvailableError,
    check_llm,
    evidence_scoring_agent,
    get_llm_config,
    load_ontology_routing_results,
    load_symptom_tree_results,
    print_pipeline_banner,
    save_evidence_scoring_results,
)

print_pipeline_banner()
LLM_CONFIG = get_llm_config()
ok, model_info = check_llm(LLM_CONFIG)
if not ok:
    raise LLMNotAvailableError(model_info)
print(f"LLM ready — {LLM_CONFIG.method_prefix()}: {model_info}")

tree_results_df, _ = load_symptom_tree_results()
routing_df = load_ontology_routing_results()
merged = tree_results_df.merge(
    routing_df, on=["patient_id", "hadm_id"], how="inner", suffixes=("", "_routing")
)
print(f"Admissions to score: {len(merged)}")

## Score Candidates Against Evidence

In [ ]:
scored = []

for _, row in merged.iterrows():
    hadm_id = str(row["hadm_id"])
    patient_id = str(row["patient_id"])
    print(f"Evidence scoring patient={patient_id} hadm_id={hadm_id}...")
    scoring = evidence_scoring_agent(
        ontology_routing=row["ontology_routing"],
        symptom_tree=row["symptom_tree"],
        extracted=row["extracted"],
        admission_id=hadm_id,
        patient_id=patient_id,
        config=LLM_CONFIG,
    )
    scored_candidates = scoring.get("scored_candidates") or []
    top = scored_candidates[0] if scored_candidates else {}
    scored.append({
        "patient_id": patient_id,
        "admission_id": row["admission_id"],
        "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]),
        "primary_icd_code": row["primary_icd_code"],
        "primary_dx_title": row["primary_dx_title"],
        "ground_truth_icd10": row["ground_truth_icd10"],
        "ground_truth_dx_titles": row["ground_truth_dx_titles"],
        "evidence_scoring_method": scoring.get("_method", "unknown"),
        "evidence_scoring": scoring,
        "n_scored": len(scored_candidates),
        "top_diagnosis": top.get("disease"),
        "top_score": top.get("score"),
    })

results_df = pd.DataFrame(scored)
results_df[["patient_id", "admission_id", "evidence_scoring_method", "n_scored", "top_diagnosis", "top_score"]]

## Save Results

In [ ]:
out = save_evidence_scoring_results(results_df)
print(f"Saved → {out}")